In [3]:
# ------ Install dependency and import libraries needed ------
#!pip install requests numpy polars pandas matplotlib
import io
import os
import re
import requests
import math
from datetime import datetime, timedelta
from zoneinfo import ZoneInfo
from pathlib import Path
import numpy as np
import polars as pl
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import subprocess

import urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# general settings for polars
pl.Config.set_tbl_cols(1000)
pl.Config.set_tbl_width_chars(1000) 

polars.config.Config

# LLM inference
This notebook will produce the table and analysis about energy consumer while inferencing models. It require the file `pue_10m.parquet` to be present. 

In [4]:
PRJ_ROOT_DIR = Path(
    subprocess.check_output(
        ["git", "rev-parse", "--show-toplevel"],
        text=True
    ).strip()
)
DATA_DIR = Path(PRJ_ROOT_DIR, "data")
print(f"Project data directory detected: {DATA_DIR}")

Project data directory detected: /orfeo/cephfs/scratch/area/ntosato/infrastructure-level-data-collection/data


In [5]:
SLURM_JOB_IDS      =[ str(i) for i in [1035875,1035945,1035952,1035958,1036052,1036071,1036203,1036165]  ]
print(SLURM_JOB_IDS)

['1035875', '1035945', '1035952', '1035958', '1036052', '1036071', '1036203', '1036165']


In [6]:
experiments_df = pl.read_parquet(f"{DATA_DIR}/experiments/experiments.parquet").filter(pl.col("SLURM_JOB_ID").is_in(SLURM_JOB_IDS))

In [7]:

# ----- CONSTANTS -----
OFFSET_IN_SECONDS = 60              # padding added around each experiment window
BUCKET_S          = 1               # aggregation bucket width for plots (seconds)

runs = (
    experiments_df
    .pivot(
        values="timestamp",
        index=["SLURM_JOB_ID", "notes"],
        on="status",
        aggregate_function="first",
    )
    .rename({"START": "start", "END": "end"})
    .filter(pl.col("start").is_not_null() & pl.col("end").is_not_null())
    .sort("start")
)

print(runs)

shape: (8, 4)
┌──────────────┬────────────────────────┬────────────────────────────────┬────────────────────────────────┐
│ SLURM_JOB_ID ┆ notes                  ┆ start                          ┆ end                            │
│ ---          ┆ ---                    ┆ ---                            ┆ ---                            │
│ str          ┆ str                    ┆ datetime[μs, UTC]              ┆ datetime[μs, UTC]              │
╞══════════════╪════════════════════════╪════════════════════════════════╪════════════════════════════════╡
│ 1035875      ┆ GPT-OSS-120B           ┆ 2026-03-27 14:32:00.987436 UTC ┆ 2026-03-27 14:47:00.843301 UTC │
│ 1035945      ┆ GPT-OSS-120B           ┆ 2026-03-27 14:55:37.466917 UTC ┆ 2026-03-27 15:10:37.352568 UTC │
│ 1035952      ┆ GPT-OSS-120B           ┆ 2026-03-27 15:16:44.405283 UTC ┆ 2026-03-27 15:31:43.757850 UTC │
│ 1035958      ┆ Llama-4-Scout-17B      ┆ 2026-03-27 15:58:06.454721 UTC ┆ 2026-03-27 16:13:06.330370 UTC │
│ 1036052     

In [8]:
# Read ipmi_power, llm_api_usage, nvidia_smi, turbostat, pue.
def read_data_from_parquet(table_name, timefilter=None, column_names=None, data_dir=DATA_DIR):
    parquet_files = sorted((data_dir / table_name).glob("*.parquet"))

    if not parquet_files:
        print(f"No parquet files found for table {table_name} in {data_dir / table_name}")
        return pl.DataFrame()

    if timefilter is not None:
        time_min, time_max = timefilter

        selected_files = []
        for file in parquet_files:
            parts = file.stem.split("_")
            file_start = datetime.fromisoformat(parts[-2]).replace(tzinfo=time_min.tzinfo)
            file_end = datetime.fromisoformat(parts[-1]).replace(tzinfo=time_min.tzinfo)

            if file_start <= time_max and file_end >= time_min:
                selected_files.append(file)

        parquet_files = selected_files

    if not parquet_files:
        print(f"No parquet files overlap requested timerange for table {table_name}")
        return pl.DataFrame()

    dfs = []
    for file in parquet_files:
        try:
            df = pl.read_parquet(file)
            if column_names:
                df = df.select(column_names)
            dfs.append(df)
        except Exception as e:
            print(f"Error reading {file}: {e}")

    if not dfs:
        print(f"No valid parquet files found for table {table_name} in {data_dir / table_name}")
        return pl.DataFrame()

    df = pl.concat(dfs, how="vertical")

    if timefilter is not None:
        df = df.filter(pl.col("timestamp").is_between(timefilter[0], timefilter[1]))

    return df

In [9]:
time_min = runs["start"].min()
time_max = runs["end"].max()

In [10]:
ipmi_power = read_data_from_parquet("ipmi_power",timefilter=(time_min,time_max)).filter(pl.col("host") == "dgx005.hpc.rd.areasciencepark.it").filter(pl.col("psu_id")=="0").filter(pl.col("unit")=="W")
turbostat = read_data_from_parquet("turbostat",timefilter=(time_min,time_max),column_names=["timestamp","core", "cpu", "die", "package_power_watt","ram_power_watt","package","host"]).filter(pl.col("host") == "dgx005.hpc.rd.areasciencepark.it")
nvidia_smi = read_data_from_parquet("nvidia_smi",timefilter=(time_min,time_max),column_names=["timestamp","host","name","index","utilization_gpu","power_draw"]).filter(pl.col("host") == "dgx005.hpc.rd.areasciencepark.it")
llm_api_usage = read_data_from_parquet("llm_api_usage",timefilter=(time_min,time_max))

In [12]:
# Example: integrate GPU power draw over time (nvidia-smi table)
# Requires columns: timestamp (datetime) and power_draw (W)

def integrate_power_to_energy(df: pl.DataFrame, ts_col: str = "timestamp", p_col: str = "power_draw"):
    if df.is_empty():
        raise ValueError("Input DataFrame is empty")

    # Keep only valid rows, sort in time order, and ensure positive dt
    w = (
        df
        .select([ts_col, p_col])
        .drop_nulls([ts_col, p_col])
        .sort(ts_col)
        .with_columns([
            pl.col(ts_col).diff().dt.total_milliseconds().alias("dt_ms"),
            pl.col(p_col).shift(1).alias("p_prev_W"),
        ])
        .with_columns([
            (pl.col("dt_ms") / 1000.0).alias("dt_s"),
            ((pl.col(p_col) + pl.col("p_prev_W"))/2 ).alias("p_avg_W"),
        ])
        .filter(pl.col("dt_s").is_not_null() & (pl.col("dt_s") > 0))
        .with_columns([
            # Joules = W * s
            (pl.col("p_avg_W") * pl.col("dt_s")).alias("dE_J"),
            # Wh = J / 3600
            (pl.col("p_avg_W") * pl.col("dt_s") / 3600.0).alias("dE_Wh"),
            (pl.col("p_avg_W") * pl.col("dt_s") / 3_600_000.0).alias("dE_kWh"),
        ])
    )

    totals = w.select([
        pl.col("dE_J").sum().alias("energy_J"),
        pl.col("dE_Wh").sum().alias("energy_Wh"),
        pl.col("dE_kWh").sum().alias("energy_kWh"),
    ])

    return w, totals
import polars as pl

def integrate_power_to_energy_ipmi(df: pl.DataFrame, ts_col: str = "timestamp", p_col: str = "value", host_col: str = "host"):
    if df.is_empty():
        raise ValueError("Input DataFrame is empty")

    # Keep only valid rows, sort by host AND time order
    w = (
        df
        .select([host_col, ts_col, p_col])
        .drop_nulls([ts_col, p_col])
        .sort([host_col, ts_col])
        .with_columns([
            # Use .over() to calculate differences and shifts PER HOST
            pl.col(ts_col).diff().dt.total_milliseconds().over(host_col).alias("dt_ms"),
            pl.col(p_col).shift(1).over(host_col).alias("p_prev_W"),
        ])
        .with_columns([
            (pl.col("dt_ms") / 1000.0).alias("dt_s"),
            ((pl.col(p_col) + pl.col("p_prev_W")) / 2.0).alias("p_avg_W"),
        ])
        .filter(pl.col("dt_s").is_not_null() & (pl.col("dt_s") > 0))
        .with_columns([
            # Joules = W * s
            (pl.col("p_avg_W") * pl.col("dt_s")).alias("dE_J"),
            # Wh = J / 3600
            (pl.col("p_avg_W") * pl.col("dt_s") / 3600.0).alias("dE_Wh"),
            (pl.col("p_avg_W") * pl.col("dt_s") / 3_600_000.0).alias("dE_kWh"),
        ])
    )

    # Group by the host column so you get a total energy readout per machine
    totals = w.group_by(host_col).agg([
        pl.col("dE_J").sum().alias("energy_J"),
        pl.col("dE_Wh").sum().alias("energy_Wh"),
        pl.col("dE_kWh").sum().alias("energy_kWh"),
    ])

    return w, totals

import polars as pl

def integrate_turbostat_energy(
    df: pl.DataFrame, 
    ts_col: str = "timestamp", 
    host_col: str = "host",
    pkg_p_col: str = "package_power_watt", 
    ram_p_col: str = "ram_power_watt",
    die_col: str = "die" # or "package"/"socket" depending on your exact schema
):la parte importante a mio parere non è salvare i dati, ma recuperarli, fare in modo che abbia i dati necessari per quel messaggio senza avere un context che lo faccia deviare dalla conversazione, quindi l'ordine delle cose, specificarli che sono "ricordi" e che tipo di peso hanno quei ricordi, un ricordo recente ha più peso, un ricordo emotivo ha più peso, ecc ecc
    if df.is_empty():
        raise ValueError("Input DataFrame is empty")

    # 1. Filter to only the cumulative rows where socket/die is null or "-"
    # Turbostat rolls up cumulative statistics on rows where core/cpu/die/package are "-"
    filtered_df = df.filter(
        pl.col(die_col) == "-"     )

    # 2. Keep only valid rows, sort by host AND time order
    w = (
        filtered_df
        .select([host_col, ts_col, pkg_p_col, ram_p_col])
        .drop_nulls([ts_col, pkg_p_col, ram_p_col])
        .sort([host_col, ts_col])
        .with_columns([
            # Use .over() to calculate differences and shifts PER HOST
            pl.col(ts_col).diff().dt.total_milliseconds().over(host_col).alias("dt_ms"),
            
            # Get previous power values for trapezoidal rule
            pl.col(pkg_p_col).shift(1).over(host_col).alias("pkg_p_prev_W"),
            pl.col(ram_p_col).shift(1).over(host_col).alias("ram_p_prev_W"),
        ])
        .with_columns([
            (pl.col("dt_ms") / 1000.0).alias("dt_s"),
            ((pl.col(pkg_p_col) + pl.col("pkg_p_prev_W")) / 2.0).alias("pkg_p_avg_W"),
            ((pl.col(ram_p_col) + pl.col("ram_p_prev_W")) / 2.0).alias("ram_p_avg_W"),
        ])
        .filter(pl.col("dt_s").is_not_null() & (pl.col("dt_s") > 0))
        .with_columns([
            # Package Joules = W * s
            (pl.col("pkg_p_avg_W") * pl.col("dt_s")).alias("pkg_dE_J"),
            (pl.col("pkg_p_avg_W") * pl.col("dt_s") / 3600.0).alias("pkg_dE_Wh"),
            (pl.col("pkg_p_avg_W") * pl.col("dt_s") / 3_600_000.0).alias("pkg_dE_kWh"),
            
            # RAM Joules = W * s
            (pl.col("ram_p_avg_W") * pl.col("dt_s")).alias("ram_dE_J"),
            (pl.col("ram_p_avg_W") * pl.col("dt_s") / 3600.0).alias("ram_dE_Wh"),
            (pl.col("ram_p_avg_W") * pl.col("dt_s") / 3_600_000.0).alias("ram_dE_kWh"),
        ])
    )

    # 3. Group by the host column so you get total energy readouts per machine
    totals = w.group_by(host_col).agg([
        pl.col("pkg_dE_J").sum().alias("pkg_energy_J"),
        pl.col("pkg_dE_Wh").sum().alias("pkg_energy_Wh"),
        pl.col("pkg_dE_kWh").sum().alias("pkg_energy_kWh"),
        
        pl.col("ram_dE_J").sum().alias("ram_energy_J"),
        pl.col("ram_dE_Wh").sum().alias("ram_energy_Wh"),
        pl.col("ram_dE_kWh").sum().alias("ram_energy_kWh"),
    ])

    return w, totals




In [14]:
import polars as pl

pue = pl.read_parquet("pue_10m.parquet")
# 1) Initialize a list to hold the summary statistics for our final report

report_data = []

for exp in runs.iter_rows(named=True):
    model = exp["notes"]
    print(f"\nIntegrating energy for jobid={exp["SLURM_JOB_ID"]} and model {model} ...")
    t_start = exp["start"]
    t_end = exp["end"]
    ipmi = ipmi_power.filter(pl.col("timestamp").is_between(t_start,t_end))
    cpu = turbostat.filter(pl.col("timestamp").is_between(t_start,t_end))
    gpu = nvidia_smi.filter(pl.col("timestamp").is_between(t_start,t_end))
    df_gpu = nvidia_smi.filter(pl.col("timestamp").is_between(t_start,t_end))
    #df_tok = data["token_usage"].filter((pl.col("timestamp") >= t_start) & (pl.col("timestamp") <= t_end))
    df_machine = ipmi_power.filter(pl.col("timestamp").is_between(t_start,t_end))
    df_turbo = turbostat.filter(pl.col("timestamp").is_between(t_start,t_end))
    UTIL_ACTIVE_THRESHOLD_PCT = 1.0  # GPU is considered active if avg utilization_gpu >= this value
    total_tokens = llm_api_usage.filter(pl.col("timestamp").is_between(t_start,t_end))["total_tokens"].sum()
    active_gpu_count = 0
    avg_gpu_usage = 0.0


    avg_pue =  pue.filter((pl.col("timestamp") >= t_start) & (pl.col("timestamp") <= t_end))["ratio"].mean()
    print(f"Average PUE during experiment: {avg_pue:.3f}")

    # Initialize variables for both active and inactive GPUs
    active_gpu_count = 0
    avg_gpu_usage = 0.0
    total_energy_wh = 0.0
    
    inactive_gpu_count = 0
    avg_inactive_gpu_usage = 0.0
    inactive_total_energy_wh = 0.0

    # 2) Compute total GPU energy in Wh from power time series
    if "index" in df_gpu.columns:
        gpu_stats = (
            df_gpu
            .drop_nulls(["index", "utilization_gpu"])
            .group_by("index")
            .agg(pl.col("utilization_gpu").mean().alias("avg_util_pct"))
            .sort("index")
        )

        active_gpu_ids = (
            gpu_stats
            .filter(pl.col("avg_util_pct") >= UTIL_ACTIVE_THRESHOLD_PCT)
            ["index"].to_list()
        )
        
        inactive_gpu_ids = (
            gpu_stats
            .filter(pl.col("avg_util_pct") < UTIL_ACTIVE_THRESHOLD_PCT)
            ["index"].to_list()
        )
        
        if not active_gpu_ids:
            print(f"No active GPUs found with avg utilization >= {UTIL_ACTIVE_THRESHOLD_PCT}%")
            continue

        # --- Process Active GPUs ---
        active_gpu_count = len(active_gpu_ids)
        avg_gpu_usage = gpu_stats.filter(pl.col("index").is_in(active_gpu_ids))["avg_util_pct"].mean()

        for gpu_idx in active_gpu_ids:
            gpu_data = df_gpu.filter(pl.col("index") == gpu_idx)
            if not gpu_data.is_empty():
                _, gpu_total = integrate_power_to_energy(gpu_data, ts_col="timestamp", p_col="power_draw")
                total_energy_wh += float(gpu_total["energy_Wh"][0])

        print(f"Active GPU average utilization (%): {avg_gpu_usage:.2f}")

        # --- Process Inactive GPUs ---
        if inactive_gpu_ids:
            inactive_gpu_count = len(inactive_gpu_ids)
            avg_inactive_gpu_usage = gpu_stats.filter(pl.col("index").is_in(inactive_gpu_ids))["avg_util_pct"].mean()
            
            for gpu_idx in inactive_gpu_ids:
                gpu_data = df_gpu.filter(pl.col("index") == gpu_idx)
                if not gpu_data.is_empty():
                    _, gpu_total = integrate_power_to_energy(gpu_data, ts_col="timestamp", p_col="power_draw")
                    inactive_total_energy_wh += float(gpu_total["energy_Wh"][0])
                    
            print(f"Inactive GPU average utilization (%): {avg_inactive_gpu_usage:.2f}")
        else:
            print("No inactive GPUs found.")

    else:
        # Fallback if 'index' is not present
        active_gpu_count = 1 
        avg_gpu_usage = df_gpu["utilization_gpu"].mean() if "utilization_gpu" in df_gpu.columns else 0.0
        _, total = integrate_power_to_energy(df_gpu, ts_col="timestamp", p_col="power_draw")
        total_energy_wh = float(total["energy_Wh"][0])

    # 3) Compute total tokens from token_usage.total_tokens

    if total_tokens is None or total_tokens <= 0:
        print("total_tokens is null/zero; cannot divide by token count")
        continue

    # 4) Energy per token (GPU)
    wh_per_token_gpu = total_energy_wh / total_tokens
    inactive_wh_per_token_gpu = inactive_total_energy_wh / total_tokens

    # Compute duration in seconds to get tokens per second
    effective_duration = (t_end - t_start).total_seconds()
    tokens_per_sec = total_tokens / effective_duration if effective_duration > 0 else 0

    print(f"\nTotal Active GPU energy used for metric (Wh): {total_energy_wh:.6f}")
    print(f"Total Inactive GPU energy used (Wh):        {inactive_total_energy_wh:.6f}")
    print(f"Total tokens:                               {total_tokens:,.0f}")
    print(f"Tokens per second:                          {tokens_per_sec:,.2f} tokens/s")
    print(f"Energy Active GPU per 1k tokens:            {wh_per_token_gpu * 1000:.9f} Wh")
    print(f"Energy Inactive GPU per 1k tokens:          {inactive_wh_per_token_gpu * 1000:.9f} Wh")

    # 5) Energy per token (System/IPMI)
    detail, total_ipmi = integrate_power_to_energy_ipmi(df_machine)
    sys_total_energy_wh = total_ipmi['energy_Wh'][0]
    wh_per_token_sys = sys_total_energy_wh / total_tokens

    # Also compute CPU energy from turbostat if available
    detail, total_turbo = integrate_turbostat_energy(df_turbo)
    cpu_total_energy_wh = total_turbo['pkg_energy_Wh'][0] + total_turbo['ram_energy_Wh'][0]
    wh_per_token_cpu = cpu_total_energy_wh / total_tokens
    print("\nMachine total from IPMI energy:")
    print(f"Energy Total per 1k tokens: {wh_per_token_sys * 1000:.9f}")

    # 6) Append results to our report list
    report_data.append({
        "Model": model,
        "Active GPUs": active_gpu_count,
        "Tokens/s": tokens_per_sec,
        "Avg Active GPU Usage (%)": avg_gpu_usage,
        "Active GPU Energy/1k Token (Wh)": wh_per_token_gpu * 1000,
        "Inactive GPU Energy/1k Token (Wh)": inactive_wh_per_token_gpu * 1000,
        "CPU Energy/1k Token (Wh)": wh_per_token_cpu * 1000,
        "System Energy/1k Token (Wh)": wh_per_token_sys * 1000,
        "Adjusted Total Energy/1k Token (Wh)": wh_per_token_sys * avg_pue * 1000
    })

# 7) Output the final consolidated report
if report_data:
    report_df = pl.DataFrame(report_data)
    
    print("\n" + "="*120)
    print(" SUMMARY REPORT")
    print("="*120)
    
    # Expanding the table width config to ensure the new columns fit cleanly on screen
    with pl.Config(tbl_rows=100, float_precision=6, tbl_width_chars=120):
        print(report_df)
else:
    print("\nNo data generated for the summary report.")


Integrating energy for jobid=1035875 and model GPT-OSS-120B ...
Average PUE during experiment: 1.395
Active GPU average utilization (%): 100.00
Inactive GPU average utilization (%): 0.00

Total Active GPU energy used for metric (Wh): 244.164265
Total Inactive GPU energy used (Wh):        105.742696
Total tokens:                               1,325,388
Tokens per second:                          1,472.89 tokens/s
Energy Active GPU per 1k tokens:            0.184220972 Wh
Energy Inactive GPU per 1k tokens:          0.079782445 Wh

Machine total from IPMI energy:
Energy Total per 1k tokens: 0.583327108

Integrating energy for jobid=1035945 and model GPT-OSS-120B ...
Average PUE during experiment: 1.413
Active GPU average utilization (%): 97.99
Inactive GPU average utilization (%): 0.00

Total Active GPU energy used for metric (Wh): 356.105563
Total Inactive GPU energy used (Wh):        70.456299
Total tokens:                               1,349,374
Tokens per second:                     